#  E-Commerce AI Pipeline: Architecture Documentation

## 1. Pipeline Flow (How Data Moves)
* **Ingestion (Bronze):** Raw e-commerce interaction events (views, carts, purchases) are ingested into a Delta Table.
* **Processing & Feature Engineering (Silver):** Data is aggregated at the `user_id` level to create behavioral features. Target leakage is carefully removed.
* **Model Training & Tracking:** Machine learning models (Logistic Regression for classification, ALS for recommendations) are trained. MLflow tracks hyperparameters, metrics (AUC), and artifacts.
* **Batch Inference (Gold):** The winning model scores all users. Results are saved in a Gold Delta Table, identifying "Top Predicted Future Buyers".
* **Governance & Reliability:** Unity Catalog manages permissions, while Delta Lake's `_delta_log` enables Time Travel and disaster recovery.

## 2. Model Retraining Strategy (Keeping the AI Smart)
To prevent model degradation over time, the following retraining strategy is defined:
* **Scheduled Retraining:** The pipeline will automatically trigger a complete retrain on the 1st of every month using Databricks Workflows.
* **Data Window:** The model will always train on the last 90 days of rolling data to capture the most recent consumer trends and seasonality (e.g., holiday sales).
* **Champion/Challenger Setup:** The new model will be logged in MLflow. If its AUC score is strictly greater than the current production model, it will automatically be promoted to the 'Champion' status in the Model Registry.